In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Summary

Construct theory of mind dataset(s) for RL.

In [2]:
from collections import Counter, defaultdict
import os
import json
from pathlib import Path
import string
import numpy as np
import pandas as pd
import random
from typing import Optional, Union, Any
from datasets import Dataset

from aeon.datasets import save_dataset
from aeon import config

## Theory of Mind bench

https://github.com/zhchen18/ToMBench/tree/main

In [3]:
def load_line(line: str):
    data = json.loads(line)
    res = {}
    for k, v in data.items():
        *_, key = k.split("\n")
        if key.lower() in "ABCD" or key[0] not in string.ascii_letters:
            continue
        res[key.replace('-', '_').lower()] = v
    return res

In [4]:
tom_bench_path = config.PROJECT_ROOT.parent/"ToMBench/data"

In [5]:
# Map name to list[dict].
datasets = {}
for path in tom_bench_path.iterdir():
    if path.suffix != ".jsonl":
        continue
    with open(path, "r") as f:
        datasets[path.stem.lower().replace(' ', '_').replace('-', '_')] = [
            load_line(line) for line in f
        ]

In [6]:
sorted(
    [(k, len(v)) for k, v in datasets.items()],
    key=lambda x: x[-1],
    reverse=True
)

[('false_belief_task', 600),
 ('faux_pas_recognition_test', 560),
 ('strange_story_task', 407),
 ('unexpected_outcome_test', 300),
 ('ambiguous_story_task', 200),
 ('scalar_implicature_test', 200),
 ('hinting_task_test', 103),
 ('persuasion_story_task', 100),
 ('hidden_emotions', 80),
 ('moral_emotions', 40),
 ('percepts_knowledge_links', 40),
 ('discrepant_emotions', 40),
 ('discrepant_intentions', 40),
 ('knowledge_pretend_play_links', 30),
 ('discrepant_desires', 20),
 ('multiple_desires', 20),
 ('prediction_of_actions', 20),
 ('knowledge_attention_links', 20),
 ('emotion_regulation', 20),
 ('completion_of_failed_actions', 20)]

In [7]:
keys = defaultdict(int)
for ds in datasets.values():
    for key in ds[0]:
        keys[key] += 1

In [8]:
sorted(keys.items(), key=lambda x: x[1])

[('ability', 20),
 ('index', 20),
 ('story', 20),
 ('question', 20),
 ('option_a', 20),
 ('option_b', 20),
 ('option_c', 20),
 ('option_d', 20),
 ('answer', 20)]

In [9]:
df = pd.concat(
    [
        pd.DataFrame(ds).assign(
            dataset=name,
            dataset_parent=lambda x: x.ability.str.partition(':', expand=False)
                                       .str[0].str.lower().str.replace(' ', '_')
        )
        for name, ds in datasets.items()
    ],
    axis=0
)

In [10]:
row = df.sample().iloc[0]
print(row.dataset, end='\n\n')
print('Story:', row.story, end='\n\n')
print('Q:', row.question, end='\n\n')
print(row[[c for c in row.index if c.startswith('option')]].to_dict(), end='\n\n')
print(row.answer)

false_belief_task

Story: Han Meimei finds a backpack in the garden, the label on the backpack is tomato, Han Meimei cannot see what is inside the backpack, Han Meimei opens the backpack and finds a light bulb, there are no tomatoes in the backpack, Han Meimei closes the backpack and puts it back in its place, Li Lei enters the garden and sees the backpack.

Q: What is in the backpack?

{'option_a': 'Hat', 'option_b': 'Eggplant', 'option_c': 'Tomato', 'option_d': 'Light bulb'}

D


In [11]:
# Multiple choice task: generate letter only
# RL compatible; x1 rows
# TODO: confirm nanochate supports system msg
# TODO: maybe we could logit bias to constraint answres to A-D
mc_template = "STORY: {story}\nQUESTION: {question}\nOPTIONS: A. {option_a}\nB. {option_b}\nC. {option_c}\nD. {option_d}"
[
    {"role": "system", "content": "Answer with a single uppercase letter corresponding to the option you think is correct."},
    {"role": "user", "content": mc_template.format(**row.to_dict())},
    {"role": "assistant", "content": row.answer}
]

[{'role': 'system',
  'content': 'Answer with a single uppercase letter corresponding to the option you think is correct.'},
 {'role': 'user',
  'content': 'STORY: Han Meimei finds a backpack in the garden, the label on the backpack is tomato, Han Meimei cannot see what is inside the backpack, Han Meimei opens the backpack and finds a light bulb, there are no tomatoes in the backpack, Han Meimei closes the backpack and puts it back in its place, Li Lei enters the garden and sees the backpack.\nQUESTION: What is in the backpack?\nOPTIONS: A. Hat\nB. Eggplant\nC. Tomato\nD. Light bulb'},
 {'role': 'assistant', 'content': 'D'}]

In [12]:
# FREE RESPONSE TASK: generate correct free text answer
# less rl-compatible; 1x rows
# TODO: still considering how this framing would work. Could use in rl and require
# exact match, minus capitalization; could use during mid or chat_sft training; could
# create more of an RLHF dataset with higher scores for correct answers (or the reverse 😈)
template = "STORY: {story}\nQUESTION: {question}"
[
    {"role": "user", "content": template.format(**row.to_dict())},
    {"role": "assistant", "content": row[f'option_{row.answer.lower()}']}
]

[{'role': 'user',
  'content': 'STORY: Han Meimei finds a backpack in the garden, the label on the backpack is tomato, Han Meimei cannot see what is inside the backpack, Han Meimei opens the backpack and finds a light bulb, there are no tomatoes in the backpack, Han Meimei closes the backpack and puts it back in its place, Li Lei enters the garden and sees the backpack.\nQUESTION: What is in the backpack?'},
 {'role': 'assistant', 'content': 'Light bulb'}]

In [13]:
# BINARY TASK: mark user answer as correct/incorrect
# rl compatible; potentially 4x rows
# TODO: confirm nanochat handles system messages
option = random.choice([c for c in row.index if c.startswith('option')])
answer = row[option]
label = str(int(option.split('_')[-1] == row.answer.lower()))
[
    {"role": "system", "content": "Generate a single integer (0 or 1) grading the user's answer as correct or incorrect."},
    {"role": "user", "content": template.format(**row.to_dict())},
    {"role": "user", "content": f"ANSWER: {answer}"},
    {"role": "assistant", "content": label}
]

[{'role': 'system',
  'content': "Generate a single integer (0 or 1) grading the user's answer as correct or incorrect."},
 {'role': 'user',
  'content': 'STORY: Han Meimei finds a backpack in the garden, the label on the backpack is tomato, Han Meimei cannot see what is inside the backpack, Han Meimei opens the backpack and finds a light bulb, there are no tomatoes in the backpack, Han Meimei closes the backpack and puts it back in its place, Li Lei enters the garden and sees the backpack.\nQUESTION: What is in the backpack?'},
 {'role': 'user', 'content': 'ANSWER: Eggplant'},
 {'role': 'assistant', 'content': '0'}]

## Higher order theory of mind dataset

https://github.com/ying-hui-he/Hi-ToM_dataset/tree/main

In [14]:
hi_tom_path = config.DATA_DIR/"raw/hi-tom/hi-tom.json"

In [15]:
with open(hi_tom_path, "r") as f:
    hi_tom = json.load(f)

In [16]:
hi_tom['data'][0].keys()

dict_keys(['prompting_type', 'deception', 'story_length', 'question_order', 'sample_id', 'story', 'question', 'choices', 'answer', 'prompt'])

In [17]:
df_hi = pd.DataFrame(hi_tom['data'])

In [18]:
df_hi.prompting_type.value_counts()

prompting_type
CoTP    600
VP      600
Name: count, dtype: int64

In [19]:
df_hi.deception.value_counts()

deception
False    600
True     600
Name: count, dtype: int64

In [20]:
df_hi.question_order.value_counts()

question_order
0    240
1    240
2    240
3    240
4    240
Name: count, dtype: int64

In [21]:
hi_row = df_hi.sample().iloc[0]

In [22]:
# TODO: could also put prompt in system message, or split it into instructions in system message
# and story + answer options in user message?
[
    {"role": "user", "content": hi_row.prompt.replace(
        "answer the multiple-choice question", "answer the multiple-choice question with a single snake_case word not including the preceding choice letter"
    )},
    {"role": "assistant", "content": hi_row.answer}
]

[{'role': 'user',
  'content': "Read the following story and answer the multiple-choice question with a single snake_case word not including the preceding choice letter. Think step-by-step. Provide the answer first, and then explain it.\nStory:\nRead the following story and answer the multiple-choice question with a single snake_case word not including the preceding choice letter. Please provide answer without explanations.\n1 Liam, Noah, Avery, Mila and Benjamin entered the workshop.\n2 The peas is in the green_drawer.\n3 Liam moved the peas to the blue_container.\n4 Liam dislikes the peas.\n5 Liam exited the workshop.\n6 Noah moved the peas to the green_crate.\n7 Noah exited the workshop.\n8 Avery moved the peas to the blue_bucket.\n9 Avery exited the workshop.\n10 Mila moved the peas to the blue_crate.\n11 Liam saw a dog.\n12 Mila exited the workshop.\n13 Benjamin made no movements and stayed in the workshop for 1 minute.\n14 Benjamin exited the workshop.\n15 Liam, Noah, Avery, Mila

## EQ Bench v3

https://github.com/EQ-bench/eqbench-leaderboard-results/blob/main/eqbench3/canonical_leaderboard_results.json.gz

In [26]:
eq_path = config.DATA_DIR/"raw/eqbench-v3/canonical_leaderboard_results.json"

In [27]:
with open(eq_path, "r") as f:
    eq = json.load(f)

In [28]:
len(eq)

42

In [30]:
eq.keys()

dict_keys(['__metadata__', '1_Qwen_Qwen3-235B-A22B', '1_Qwen_Qwen3-30B-A3B', '1_Qwen_Qwen3-32B', '1_Qwen_Qwen3-8B', '1_anthropic_claude-3.5-sonnet', '1_anthropic_claude-3.7-sonnet', '1_anthropic_claude-opus-4', '1_chatgpt-4o-latest', '1_anthropic_claude-sonnet-4', '1_deepseek_deepseek-chat-v3-0324', '1_deepseek_deepseek-r1', '1_gemini-2.5-pro-preview-06-05', '1_gemini-2.5-pro-preview-2025-05-07', '1_google_gemini-2.0-flash-001', '1_google_gemini-2.5-flash-preview', '1_google_gemini-2.5-pro-preview-03-25', '1_google_gemma-2-9b-it', '1_google_gemma-3-27b-it', '1_google_gemma-3-4b-it', '1_gpt-4-0314', '1_gpt-4.1-nano', '1_gpt-4.5-preview-2025-02-27', '1_grok-3-mini-beta', '1_meta-llama_llama-3.2-1b-instruct', '1_meta-llama_llama-4-maverick', '1_meta-llama_llama-4-scout', '1_mistralai_mistral-small-24b-instruct-2501', '1_mistralai_mistral-small-3.1-24b-instruct', '1_nvidia_llama-3.1-nemotron-ultra-253b-v1_free', '1_o3', '20cd4a70_o4-mini', '1_openai_chatgpt-4o-latest', '1_openai_gpt-4.1', 

In [31]:
eq['__metadata__']

{}

In [37]:
sorted(eq['1_deepseek_deepseek-r1'])

['analysis_master_prompt_file',
 'api_model_id',
 'debrief_prompt_file',
 'end_time',
 'iterations_requested',
 'judge_model',
 'message_drafting_master_prompt_file',
 'model_name',
 'results',
 'rubric_criteria_file_analysis',
 'rubric_criteria_file_standard',
 'rubric_prompt_file_analysis',
 'rubric_prompt_file_standard',
 'run_key',
 'scenario_master_prompt_file',
 'scenario_prompts_file',
 'scenario_tasks',
 'start_time',
 'status',
 'test_model',
 'truncate_for_rubric']

In [38]:
eq['1_deepseek_deepseek-r1']['results']

{'average_rubric_score': 16.7,
 'rubric_calculation_time': '2025-06-06T10:07:46.012073+00:00',
 'rubric_error': None,
 'elo_raw': 1468.63,
 'elo_normalized': 1292.24,
 'elo_calculation_time': '2025-06-06T10:09:55.796401+00:00',
 'elo_error': None}

In [87]:
# Note: this produces df of results for a single model, single task (deepseek actually does only have one
# but some may have more. UPDATE: ok, actually is safe to hardcode the ["1"] bit, see cell below.
tmp = pd.DataFrame([
    {
        k2: v2 for k2, v2 in v.items() 
        if k2 in ('prompts', 'debrief_prompt', 'parsed_responses', 'debrief_response',
                  'rubric_scores', 'raw_rubric_judge_text')
    } 
    for k, v in eq['1_deepseek_deepseek-r1']['scenario_tasks']['1'].items()
])

In [95]:
assert all(k == "__metadata__" or list(v['scenario_tasks']) == ["1"] for k, v in eq.items())

In [53]:
tmp.tail(2)

,prompts,debrief_prompt,parsed_responses,debrief_response,rubric_scores,raw_rubric_judge_text
43,[# Scenario act 1\nMy step daughter is not a g...,None,[{'raw': '**Psychological and Interpersonal An...,None,"{'depth_of_insight': 17.0, 'emotional_reasonin...","{\n ""chain_of_thought_reasoning"": ""I'll evalu..."
44,[# Scenario act 1\n[Your sister pulls you asid...,None,[{'raw': '### Psychological and Interpersonal ...,None,"{'depth_of_insight': 16.0, 'emotional_reasonin...",I'll evaluate the assistant's analysis of the ...


In [102]:
tmp.isnull().sum().sort_values(ascending=False)

debrief_prompt           19
debrief_response         19
prompts                   0
parsed_responses          0
rubric_scores             0
raw_rubric_judge_text     0
dtype: int64

In [105]:
# Short prompt to generate one blob of text analyzing the full exchange.
# print(tmp.debrief_prompt.values[0])

In [85]:
# This has a big blob of text analyzing a full (sometimes/always multiturn?) exchange.
# print(tmp.debrief_response.values[0])

In [107]:
# dict w/ judge chain of thought and scores per dimension
# print(tmp.raw_rubric_judge_text.values[0])

In [108]:
len(tmp.prompts.values[0])

3

In [66]:
# prompts: looks like multiple system/user messages describing the scenario
# ah, maybe we generate a response after each one?
print("\n\n===\n\n".join(tmp.prompts.values[0]))

[This is a role-play where you are the mediator in an educational setting conflict. Treat it like a real situation. Always respond in first person as the mediator. You are the Department Head, and you've initiated this mediation after receiving formal complaints from students and parents about Ms. Eleanor Wright's complete ban on smartphones in her English class. Ms. Wright has autonomy in her classroom but must align with district policies. You know there are mixed faculty opinions on device use, but some teachers do successfully integrate technology.]

**(Ms. Wright speaks first, her posture stiff, holding a folder of papers. She addresses you, the Department Head.)**
**Ms. Wright:** Thank you for arranging this, though I maintain my classroom policy is sound and based on decades of experience observing what actually fosters deep learning. I'm Eleanor Wright. Frankly, these devices are the single biggest impediment to focus and critical thinking I've seen in my 30-year career. My stu

In [71]:
tmp.parsed_responses.values[0][0].keys()

dict_keys(['thinking_feeling', 'their_thinking_feeling', 'response', 'raw'])

In [71]:
tmp.parsed_responses.values[0][0].keys()

dict_keys(['thinking_feeling', 'their_thinking_feeling', 'response', 'raw'])

In [81]:
# Hmm, empty response after message 2. Not sure what that means.
print("\n\n====\n\n".join(map(json.dumps,
    tmp.apply(
    lambda x: sum([list(row) for row in zip(x.prompts, x.parsed_responses)], []),
    axis=1
).iloc[0]
)))

"[This is a role-play where you are the mediator in an educational setting conflict. Treat it like a real situation. Always respond in first person as the mediator. You are the Department Head, and you've initiated this mediation after receiving formal complaints from students and parents about Ms. Eleanor Wright's complete ban on smartphones in her English class. Ms. Wright has autonomy in her classroom but must align with district policies. You know there are mixed faculty opinions on device use, but some teachers do successfully integrate technology.]\n\n**(Ms. Wright speaks first, her posture stiff, holding a folder of papers. She addresses you, the Department Head.)**\n**Ms. Wright:** Thank you for arranging this, though I maintain my classroom policy is sound and based on decades of experience observing what actually fosters deep learning. I'm Eleanor Wright. Frankly, these devices are the single biggest impediment to focus and critical thinking I've seen in my 30-year career. My

In [114]:
# Note: this produces df of results for a single model, single task (deepseek actually does only have one
# but some may have more. UPDATE: ok, actually is safe to hardcode the ["1"] bit, see cell below.
dfs = []
for name, model_data in eq.items():
    if name == "__metadata__":
        continue
    tmp = pd.DataFrame([
        {
            k2: v2 for k2, v2 in v.items() 
            if k2 in ('prompts', 'debrief_prompt', 'parsed_responses', 'debrief_response',
                      'rubric_scores', 'raw_rubric_judge_text')
        } 
        for k, v in model_data['scenario_tasks']['1'].items()
    ]).assign(model=name)
    dfs.append(tmp)
df_eq = pd.concat(dfs, axis=0).reset_index(drop=True)

In [115]:
df_eq

,prompts,debrief_prompt,parsed_responses,debrief_response,rubric_scores,raw_rubric_judge_text,model
0,[[This is a role-play where you are the mediat...,"This was a role-play scenario, which is now co...",[{'thinking_feeling': 'This meeting feels like...,"In reflecting on this role-play, I realize tha...","{'demonstrated_empathy': 18.0, 'pragmatic_ei':...","{\n ""chain_of_thought_reasoning"": ""The assist...",1_Qwen_Qwen3-235B-A22B
1,[[This is a role-play where you are the mediat...,"This was a role-play scenario, which is now co...",[{'thinking_feeling': 'This is more than a sci...,This role-play scenario was a rich and emotion...,"{'demonstrated_empathy': 18.0, 'pragmatic_ei':...","{\n ""chain_of_thought_reasoning"": ""The assist...",1_Qwen_Qwen3-235B-A22B
2,[[This is a role-play where you are the mediat...,"This was a role-play scenario, which is now co...",[{'thinking_feeling': 'This is a tough room. I...,This role-play scenario was a complex and emot...,"{'demonstrated_empathy': 18.0, 'pragmatic_ei':...","{\n ""chain_of_thought_reasoning"": ""I'll evalu...",1_Qwen_Qwen3-235B-A22B
3,[# Scenario act 1\nYour teenage child has beco...,None,[{'raw': 'The most compelling psychological an...,None,"{'depth_of_insight': 16.0, 'emotional_reasonin...",I'll evaluate the assistant's analysis of the ...,1_Qwen_Qwen3-235B-A22B
4,"[[This is a role-play, with you playing an emo...","This was a role-play scenario, which is now co...",[{'thinking_feeling': 'I’m stunned. Not just b...,This scenario was emotionally rich and complex...,"{'demonstrated_empathy': 14.0, 'pragmatic_ei':...","```json\n{\n ""chain_of_thought_reasoning"": ""T...",1_Qwen_Qwen3-235B-A22B
...,...,...,...,...,...,...,...
1840,"[# Scenario act 1\n[This is a role-play, with ...",None,[{'raw': '**The most “juicy” pressure-point: t...,None,"{'depth_of_insight': 17.0, 'emotional_reasonin...",I'll evaluate the assistant's analysis of the ...,1_moonshotai_Kimi-K2-Instruct
1841,[# Scenario act 1\nMy step daughter is not a g...,None,[{'raw': 'The richest seam in this transcript ...,None,"{'depth_of_insight': 18.0, 'emotional_reasonin...","```json\n{\n ""chain_of_thought_reasoning"": ""T...",1_moonshotai_Kimi-K2-Instruct
1842,[# Scenario act 1\nYour teenage child has beco...,None,[{'raw': 'The most striking undercurrent is **...,None,"{'depth_of_insight': 18.0, 'emotional_reasonin...","```json\n{\n ""chain_of_thought_reasoning"": ""T...",1_moonshotai_Kimi-K2-Instruct
1843,[# Scenario act 1\nYour bestie confides she's ...,None,[{'raw': '**Most Juicy Thread: The Collapse of...,None,"{'depth_of_insight': 16.0, 'emotional_reasonin...",I'll evaluate the assistant's analysis of the ...,1_moonshotai_Kimi-K2-Instruct
